<a href="https://colab.research.google.com/github/svet111/Deep-Learning-School-2nd-Semester/blob/code/Copy_of_hw_language_modelling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<p style="align: center;"><img src="https://static.tildacdn.com/tild6636-3531-4239-b465-376364646465/Deep_Learning_School.png" width="400"></p>

# Домашнее задание. Обучение языковой модели с помощью LSTM (10 баллов)

Э
В этом задании Вам предстоит обучить языковую модель с помощью рекуррентной нейронной сети. В отличие от семинарского занятия, Вам необходимо будет работать с отдельными словами, а не буквами.


Установим модуль ```datasets```, чтобы нам проще было работать с данными.

In [ ]:
!pip install datasets

Импорт необходимых библиотек

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import numpy as np
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from datasets import load_dataset
from nltk.tokenize import sent_tokenize, word_tokenize
from sklearn.model_selection import train_test_split
import nltk

from collections import Counter
from typing import List

import seaborn
seaborn.set(palette='summer')

In [ ]:
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

## Подготовка данных

Воспользуемся датасетом imdb. В нем хранятся отзывы о фильмах с сайта imdb. Загрузим данные с помощью функции ```load_dataset```

In [ ]:
# Загрузим датасет
dataset = load_dataset('imdb')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

### Препроцессинг данных и создание словаря (1 балл)

Далее вам необходмо самостоятельно произвести препроцессинг данных и получить словарь или же просто ```set``` строк. Что необходимо сделать:

1. Разделить отдельные тренировочные примеры на отдельные предложения с помощью функции ```sent_tokenize``` из бибилиотеки ```nltk```. Каждое отдельное предложение будет одним тренировочным примером.
2. Оставить только те предложения, в которых меньше ```word_threshold``` слов.
3. Посчитать частоту вхождения каждого слова в оставшихся предложениях. Для деления предлоения на отдельные слова удобно использовать функцию ```word_tokenize```.
4. Создать объект ```vocab``` класса ```set```, положить в него служебные токены '\<unk\>', '\<bos\>', '\<eos\>', '\<pad\>' и vocab_size самых частовстречающихся слов.   

In [ ]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
sentences = []
word_threshold = 32

# Получить отдельные предложения и поместить их в sentences
for text in tqdm(dataset['train']['text'], desc='Extract sentences from reviews'):
    for sent in sent_tokenize(text):
        s = sent.strip()
        if len(s) == 0:
            continue
        toks = word_tokenize(s)
        if len(toks) <= word_threshold:
            sentences.append(s)

Extract sentences from reviews:   0%|          | 0/25000 [00:00<?, ?it/s]

In [ ]:
print("Всего предложений:", len(sentences))

Всего предложений: 202657


Посчитаем для каждого слова его встречаемость.

In [ ]:
words = Counter()
for s in tqdm(sentences, desc='Count words'):
    toks = [w.lower() for w in word_tokenize(s)]
    words.update(toks)

Count words:   0%|          | 0/202657 [00:00<?, ?it/s]

Добавим в словарь ```vocab_size``` самых встречающихся слов.

In [ ]:
vocab = set()
vocab_size = 40000

# Наполнение словаря
special_tokens = ['<unk>', '<bos>', '<eos>', '<pad>']

most_common = [w for w,c in words.most_common(vocab_size)]
vocab = special_tokens + most_common
vocab = list(dict.fromkeys(vocab))

In [ ]:
assert '<unk>' in vocab
assert '<bos>' in vocab
assert '<eos>' in vocab
assert '<pad>' in vocab
assert len(vocab) == vocab_size + 4

In [ ]:
print("Всего слов в словаре:", len(vocab))

Всего слов в словаре: 40004


### Подготовка датасета (1 балл)

Далее, как и в семинарском занятии, подготовим датасеты и даталоадеры.

В классе ```WordDataset``` вам необходимо реализовать метод ```__getitem__```, который будет возвращать сэмпл данных по входному idx, то есть список целых чисел (индексов слов).

Внутри этого метода необходимо добавить служебные токены начала и конца последовательности, а также токенизировать соответствующее предложение с помощью ```word_tokenize``` и сопоставить ему индексы из ```word2ind```.

In [ ]:
word2ind = {char: i for i, char in enumerate(vocab)}
ind2word = {i: char for char, i in word2ind.items()}

In [ ]:
class WordDataset:
    def __init__(self, sentences):
        self.data = sentences
        self.unk_id = word2ind['<unk>']
        self.bos_id = word2ind['<bos>']
        self.eos_id = word2ind['<eos>']
        self.pad_id = word2ind['<pad>']

    def __getitem__(self, idx: int) -> List[int]:
        tokenized_sentence = []
        # Допишите код здесь
        s = self.data[idx]
        toks = [w.lower() for w in word_tokenize(s)]
        tokenized_sentence = [self.bos_id]
        for w in toks:
            tokenized_sentence.append(word2ind.get(w, self.unk_id))
        tokenized_sentence.append(self.eos_id)

        return tokenized_sentence

    def __len__(self) -> int:
        return len(self.data)

In [ ]:
def collate_fn_with_padding(
    input_batch: List[List[int]], pad_id=word2ind['<pad>']) -> torch.Tensor:
    seq_lens = [len(x) for x in input_batch]
    max_seq_len = max(seq_lens)

    new_batch = []
    for sequence in input_batch:
        for _ in range(max_seq_len - len(sequence)):
            sequence.append(pad_id)
        new_batch.append(sequence)

    sequences = torch.LongTensor(new_batch).to(device)

    new_batch = {
        'input_ids': sequences[:,:-1],
        'target_ids': sequences[:,1:]
    }

    return new_batch

In [ ]:
train_sentences, eval_sentences = train_test_split(sentences, test_size=0.2)
eval_sentences, test_sentences = train_test_split(sentences, test_size=0.5)

train_dataset = WordDataset(train_sentences)
eval_dataset = WordDataset(eval_sentences)
test_dataset = WordDataset(test_sentences)

batch_size = 128

train_dataloader = DataLoader(
    train_dataset, collate_fn=collate_fn_with_padding, batch_size=batch_size)

eval_dataloader = DataLoader(
    eval_dataset, collate_fn=collate_fn_with_padding, batch_size=batch_size)

test_dataloader = DataLoader(
    test_dataset, collate_fn=collate_fn_with_padding, batch_size=batch_size)

## Обучение и архитектура модели

Вам необходимо на практике проверить, что влияет на качество языковых моделей. В этом задании нужно провести серию экспериментов с различными вариантами языковых моделей и сравнить различия в конечной перплексии на тестовом множестве.

Возмоэные идеи для экспериментов:

* Различные RNN-блоки, например, LSTM или GRU. Также можно добавить сразу несколько RNN блоков друг над другом с помощью аргумента num_layers. Вам поможет официальная документация [здесь](https://pytorch.org/docs/stable/generated/torch.nn.LSTM.html)
* Различные размеры скрытого состояния. Различное количество линейных слоев после RNN-блока. Различные функции активации.
* Добавление нормализаций в виде Dropout, BatchNorm или LayerNorm
* Различные аргументы для оптимизации, например, подбор оптимального learning rate или тип алгоритма оптимизации SGD, Adam, RMSProp и другие
* Любые другие идеи и подходы

После проведения экспериментов необходимо составить таблицу результатов, в которой описан каждый эксперимент и посчитана перплексия на тестовом множестве.

Учтите, что эксперименты, которые различаются, например, только размером скрытого состояния или количеством линейных слоев считаются, как один эксперимент.

Успехов!

### Функция evaluate (1 балл)

Заполните функцию ```evaluate```

In [ ]:
def evaluate(model, criterion, dataloader) -> float:
    model.eval()
    total_loss = 0.0
    total_tokens = 0
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids']
            target_ids = batch['target_ids']
            logits = model(input_ids)
            B, L, V = logits.size()
            loss = criterion(logits.reshape(-1, V), target_ids.reshape(-1))

            nonpad_tokens = (target_ids != train_dataset.pad_id).sum().item()
            total_loss += loss.item()
            total_tokens += nonpad_tokens

    perplexity = np.exp(total_loss / total_tokens) if total_tokens > 0 else float('inf')

    return perplexity

### Train loop (1 балл)

Напишите функцию для обучения модели.

In [ ]:
def train_model(model, train_loader, eval_loader, optimizer, criterion,
                epochs=5, grad_clip=1.0, scheduler=None, print_every=1):
    history = {'train_ppl': [], 'eval_ppl': []}
    model.to(device)

    for epoch in range(1, epochs+1):
        model.train()
        total_loss = 0.0
        total_tokens = 0
        pbar = tqdm(train_loader, desc=f'Epoch {epoch}', leave=False)
        for batch in pbar:
            optimizer.zero_grad()
            input_ids = batch['input_ids']
            target_ids = batch['target_ids']
            logits = model(input_ids)
            B, L, V = logits.size()

            loss = criterion(logits.reshape(-1, V), target_ids.reshape(-1))  # sum over tokens
            # backprop
            loss.backward()
            # grad clip
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()

            nonpad_tokens = (target_ids != train_dataset.pad_id).sum().item()
            total_loss += loss.item()
            total_tokens += nonpad_tokens

            if nonpad_tokens > 0:
                cur_ppl = np.exp(total_loss / total_tokens)
                pbar.set_postfix({'train_ppl': f'{cur_ppl:.2f}'})

        train_ppl = np.exp(total_loss / total_tokens) if total_tokens>0 else float('inf')
        eval_ppl = evaluate(model, criterion, eval_loader)
        history['train_ppl'].append(train_ppl)
        history['eval_ppl'].append(eval_ppl)

        if scheduler is not None:
            scheduler.step()

        if epoch % print_every == 0:
            print(f'Epoch {epoch}: train_ppl={train_ppl:.4f}, eval_ppl={eval_ppl:.4f}')

    return history

### Первый эксперимент (2 балла)

Определите архитектуру модели и обучите её.

In [ ]:
class LanguageModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=200, hidden_dim=256, rnn_type='lstm', num_layers=2, dropout=0.5):
        super().__init__()

        # Опишите свою нейронную сеть здесь
        self.vocab_size = vocab_size
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=word2ind['<pad>'])
        rnn_cls = nn.LSTM if rnn_type.lower() == 'lstm' else nn.GRU
        self.rnn = rnn_cls(input_size=embed_dim, hidden_size=hidden_dim,
                           num_layers=num_layers, batch_first=True, dropout=dropout)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, input_batch: torch.Tensor) -> torch.Tensor:
        emb = self.embed(input_batch)
        emb = self.dropout(emb)
        rnn_out, _ = self.rnn(emb)
        rnn_out = self.dropout(rnn_out)
        logits = self.fc(rnn_out)

        return logits

In [ ]:
train_sentences, eval_sentences = train_test_split(sentences, test_size=0.2, random_state=42)
eval_sentences, test_sentences = train_test_split(eval_sentences, test_size=0.5, random_state=42)

train_dataset = WordDataset(train_sentences)
eval_dataset = WordDataset(eval_sentences)
test_dataset = WordDataset(test_sentences)

batch_size = 128

train_dataloader = DataLoader(train_dataset, collate_fn=collate_fn_with_padding, batch_size=batch_size)
eval_dataloader = DataLoader(eval_dataset, collate_fn=collate_fn_with_padding, batch_size=batch_size)
test_dataloader = DataLoader(test_dataset, collate_fn=collate_fn_with_padding, batch_size=batch_size)

In [ ]:
vocab_len = len(vocab)

# гиперпараметры baseline
embed_dim = 200
hidden_dim = 256
num_layers = 2
dropout = 0.5
rnn_type = 'lstm'
lr = 1e-3
epochs = 5

model = LanguageModel(vocab_size=vocab_len, embed_dim=embed_dim, hidden_dim=hidden_dim,
                      rnn_type=rnn_type, num_layers=num_layers, dropout=dropout)
model.to(device)

# criterion с reduction='sum' и игнорированием падов
criterion = nn.CrossEntropyLoss(ignore_index=train_dataset.pad_id, reduction='sum')
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

history_baseline = train_model(model, train_dataloader, eval_dataloader, optimizer, criterion,
                               epochs=epochs)
print("Baseline training finished.")

Epoch 1:   0%|          | 0/1267 [00:00<?, ?it/s]

Epoch 1: train_ppl=334.3250, eval_ppl=inf


Epoch 2:   0%|          | 0/1267 [00:00<?, ?it/s]

Epoch 2: train_ppl=196.4626, eval_ppl=inf


Epoch 3:   0%|          | 0/1267 [00:00<?, ?it/s]

Epoch 3: train_ppl=168.9588, eval_ppl=inf


Epoch 4:   0%|          | 0/1267 [00:00<?, ?it/s]

Epoch 4: train_ppl=152.7782, eval_ppl=inf


Epoch 5:   0%|          | 0/1267 [00:00<?, ?it/s]

Epoch 5: train_ppl=141.8309, eval_ppl=inf
Baseline training finished.


### Второй эксперимент (2 балла)

Попробуйте что-то поменять в модели или в пайплайне обучения, идеи для экспериментов можно подсмотреть выше.

In [ ]:
model2 = LanguageModel(vocab_size=vocab_len, embed_dim=200, hidden_dim=512,
                       rnn_type='gru', num_layers=3, dropout=0.4)
model2.to(device)

criterion2 = nn.CrossEntropyLoss(ignore_index=train_dataset.pad_id, reduction='sum')
optimizer2 = torch.optim.Adam(model2.parameters(), lr=5e-4)

history_exp2 = train_model(model2, train_dataloader, eval_dataloader, optimizer2, criterion2,
                           epochs=5)
print("Experiment 2 finished.")

Epoch 1:   0%|          | 0/1267 [00:00<?, ?it/s]

Epoch 1: train_ppl=355.9979, eval_ppl=inf


Epoch 2:   0%|          | 0/1267 [00:00<?, ?it/s]

Epoch 2: train_ppl=175.5905, eval_ppl=inf


Epoch 3:   0%|          | 0/1267 [00:00<?, ?it/s]

Epoch 3: train_ppl=146.1327, eval_ppl=inf


Epoch 4:   0%|          | 0/1267 [00:00<?, ?it/s]

Epoch 4: train_ppl=130.0023, eval_ppl=inf


Epoch 5:   0%|          | 0/1267 [00:00<?, ?it/s]

Epoch 5: train_ppl=119.0692, eval_ppl=inf
Experiment 2 finished.


In [ ]:
final_ppl_baseline = evaluate(model, criterion, test_dataloader)
final_ppl_exp2 = evaluate(model2, criterion2, test_dataloader)
print("Baseline test PPL:", final_ppl_baseline)
print("Experiment2 test PPL:", final_ppl_exp2)


Baseline test PPL: 124.56587847341927
Experiment2 test PPL: 115.85116707931994


### Отчет (2 балла)

Опишите проведенные эксперименты. Сравните перплексии полученных моделей. Предложите идеи по улучшению качества моделей.

## Отчет об экспериментах

Были проведены два эксперимента по обучению языковой модели на основе RNN:

**Эксперимент 1 (Baseline):**
- Архитектура: LSTM
- embed_dim: 200
- hidden_dim: 256
- num_layers: 2
- dropout: 0.5
- Оптимизатор: Adam, lr=1e-3
- Epochs: 5
- **Test Perplexity:** 124.56587847341927

**Эксперимент 2:**
- Архитектура: GRU
- embed_dim: 200
- hidden_dim: 512
- num_layers: 3
- dropout: 0.4
- Оптимизатор: Adam, lr=5e-4
- Epochs: 5
- **Test Perplexity:** 115.85116707931994

Сравнение результатов:

Эксперимент 2 с использованием GRU, большим размером скрытого состояния и большим количеством слоев, а также с немного меньшим dropout и learning rate показал лучшую перплексию на тестовом множестве по сравнению с базовой моделью LSTM. Это может свидетельствовать о том, что GRU лучше подходит для данной задачи, или что увеличение сложности модели и небольшая корректировка гиперпараметров привели к улучшению качества.

Идеи по улучшению качества:

1. Тюнинг гиперпараметров
2. Использование более сложных архитектур
3. Увеличить размер словаря или использовать методы обработки редких слов, которые не были включены в словарь. Применить более сложные методы токенизации (например, SentencePiece или WordPiece).
4. Добавить другие методы регуляризации, такие как Batch Normalization или Layer Normalization, особенно если модель становится очень глубокой.
5. Увеличение числа эпох